# Kaggriculture Agent Development

**Washamba Bots** &mdash; development and evaluation notebook for `nikaangukia_meroni`,
our agent for the Kaggle Kaggriculture simulation competition.

Two agents each manage a farm across a 30-day season (720 turns) and compete for the
higher closing bank balance. There is no static train/test split: every result comes
from running live episodes.

This notebook covers the full working loop we use to develop the agent:

| Section | Purpose |
|---|---|
| 1. Environment setup | Install and pin the simulator, verify the runtime |
| 2. The simulation | Read the authoritative game parameters from the engine |
| 3. Observation schema | Inspect what the agent actually receives each turn |
| 4. Agent under test | Load the current agent from `main.py` |
| 5. Evaluation methodology | Seeded batches, and why single episodes mislead |
| 6. Results | Performance against all three reference opponents |
| 7. Behavioural diagnostics | Where turns go, and what the farm looks like at season end |
| 8. Findings | What the measurements taught us |
| 9. Next steps | Prioritised backlog |

**Prerequisite.** Select the `Python 3.13 (washamba_bots)` kernel. If it is not listed,
register it once from the project root:

```bash
.venv/Scripts/python.exe -m ipykernel install --user \
    --name washamba-bots --display-name "Python 3.13 (washamba_bots)"
```

## 1. Environment setup

The engine is patched mid-competition. A balance change in August 2026 reduced Town
Center demand and made shop unlocks sample *with* replacement, so competition staff
directed everyone to `kaggle-environments >= 1.32.6`. The official starter notebook
still pins `>= 1.32.2`, which predates that patch &mdash; do not copy it.

`%pip` installs into the *running kernel*. `!pip` shells out to whatever `pip` appears
first on `PATH`, which can silently be a different interpreter entirely.

In [ ]:
%pip install -q --upgrade "kaggle-environments>=1.32.6"

In [ ]:
import platform
from importlib.metadata import version

from kaggle_environments import make

print(f"Python              {platform.python_version()}")
print(f"kaggle-environments {version('kaggle-environments')}")

## 2. The simulation

Competition staff have been explicit that **the engine is the source of truth**: the
published documentation and the implementation have disagreed more than once over the
season, and the engine wins every time.

So rather than restating constants from the docs, we read them from the environment
that will actually score us.

In [ ]:
env = make("kaggriculture", debug=False)
config = env.configuration

PARAMETERS_OF_INTEREST = [
    ("episodeSteps", "Turns per season"),
    ("turnsPerDay", "Turns per in-game day"),
    ("boardSize", "Farm width/height in tiles"),
    ("startingMoney", "Opening bank balance"),
    ("shedCapacity", "Non-seed storage cap"),
    ("maxMarketOrdersPerTurn", "Market orders processed per turn"),
    ("actTimeout", "Seconds allowed per turn"),
    ("weedSpawnChance", "Per-tile daily weed chance"),
    ("farmHandCostMult", "Multiplier on the hire cost sequence"),
]

print(f"{'Parameter':26s} {'Value':>10s}   Meaning")
print("-" * 78)
for key, meaning in PARAMETERS_OF_INTEREST:
    print(f"{key:26s} {str(config.get(key)):>10s}   {meaning}")

Two of these constrain agent design far more than the rest.

**`actTimeout` is 1 second per turn**, with a 60-second overage bank for the whole
episode (readable at runtime as `obs["remainingOverageTime"]`). That rules out any
per-turn deep search; the agent must decide with cheap, local reasoning.

**`maxMarketOrdersPerTurn` is 10, and surplus orders are dropped silently** &mdash; no
exception, no warning. An agent that emits twelve orders simply loses two of them.

## 3. Observation schema

Each turn the agent receives a single `obs` dictionary and returns one action per unit
plus a list of market orders.

The critical detail is an axis convention that does not match between two fields:
**`tiles` is row-major (`tiles[y][x]`), while unit positions are `[x, y]`.** Indexing
one with the other produces no error &mdash; just an agent that quietly works the wrong
tile.

In [ ]:
env = make("kaggriculture", configuration={"episodeSteps": 24}, debug=False)
env.run(["starter", "starter"])

observation = env.steps[1][0].observation
farm = observation["farms"][observation["player"]]

print("Top-level observation keys:")
print("  " + ", ".join(sorted(observation.keys())))

print("\nPublic farm state (both players visible):")
for field in ["money", "farmer", "hands", "unlocked_quadrants", "hires_today"]:
    print(f"  {field:20s} {farm[field]}")

print("\nPrivate state (ours only - the opponent's shed is never visible):")
for field, contents in observation["private"].items():
    print(f"  {field:20s} {contents}")

A tile is one of five shapes, and the `kind` key must be checked before assuming any
structure: `None` (empty and unlocked), the string `"LOCKED"`, a plant dictionary, a
weed dictionary, or an animal structure (coop or pasture).

In [ ]:
from collections import Counter


def describe_tiles(farm):
    """Summarise a farm grid by tile kind."""
    kinds = Counter()
    for row in farm["tiles"]:
        for tile in row:
            if tile is None:
                kinds["empty"] += 1
            elif isinstance(tile, str):
                kinds[tile.lower()] += 1
            else:
                kinds[tile.get("kind", "unknown").lower()] += 1
    return kinds


for kind, count in describe_tiles(farm).most_common():
    print(f"  {kind:10s} {count:3d}")

## 4. Agent under test

The agent lives in `main.py` at the repository root, which is also exactly what gets
submitted to Kaggle. The notebook imports that file rather than duplicating strategy
code, so what we measure here is what competes.

One submission rule is worth stating because it fails silently: the framework selects
**the last callable in the module namespace**, not a function named `agent`. A helper
function or class defined below the agent silently becomes the submission, the episode
still reports `DONE`, and the agent finishes on exactly its starting money.

In [ ]:
import pathlib
import subprocess
import sys


def locate_repository():
    """Find main.py locally, or clone the repo when running on Kaggle/Colab."""
    for candidate in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (candidate / "main.py").exists():
            return candidate

    target = pathlib.Path("washamba_bots")
    if not target.exists():
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/Kinjuriu/washamba_bots.git"],
            check=True,
        )
    return target.resolve()


REPO = locate_repository()
AGENT_PATH = str(REPO / "main.py")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import main

print(f"Repository: {REPO}")
print(f"Submitted entry point resolves to: {main.agent.__name__}")

## 5. Evaluation methodology

**A single episode cannot distinguish an improvement from luck.** Run-to-run spread on
an identical agent has exceeded 1,400 bank on the same matchup, which is larger than
most changes we would like to detect.

Two rules follow, and both are load-bearing:

1. **Always evaluate over a batch of fixed seeds**, and report the mean and win rate
   rather than any individual score.
2. **A/B strategy changes against `pass` and `starter`.** Passing a `seed` makes the
   *environment* deterministic &mdash; weed spawns and shop unlocks &mdash; but it does
   not control the built-in `random` agent's own RNG, so that opponent still drifts
   between runs on an identical seed.

In [ ]:
import statistics


def evaluate(agent_path, opponent, seeds, episode_steps=720):
    """Play one seeded episode per seed and collect the closing balances."""
    rows = []
    for seed in seeds:
        env = make(
            "kaggriculture",
            configuration={"episodeSteps": episode_steps, "seed": seed},
            debug=False,
        )
        env.run([agent_path, opponent])
        ours, theirs = env.steps[-1]
        rows.append(
            {
                "opponent": opponent,
                "seed": seed,
                "ours": ours.reward,
                "theirs": theirs.reward,
                "won": ours.reward > theirs.reward,
            }
        )
    return rows


def summarise(rows):
    """Reduce per-episode rows to the numbers worth reporting."""
    ours = [row["ours"] for row in rows]
    return {
        "episodes": len(rows),
        "mean": round(statistics.mean(ours)),
        "stdev": round(statistics.stdev(ours)) if len(ours) > 1 else 0,
        "min": round(min(ours)),
        "max": round(max(ours)),
        "wins": sum(row["won"] for row in rows),
    }

### Running the batch

Each 720-turn episode takes roughly seven seconds. `SEEDS` below is deliberately small
so the notebook stays responsive; **use twelve or more seeds before acting on a
result.** The three built-in opponents are `pass` (does nothing, banks its opening
3,000), `random`, and `starter` (a deterministic reference agent).

In [ ]:
SEEDS = range(4)
OPPONENTS = ["pass", "random", "starter"]

results = {}
for opponent in OPPONENTS:
    rows = evaluate(AGENT_PATH, opponent, SEEDS)
    results[opponent] = summarise(rows)
    print(f"  {opponent:8s} complete")

## 6. Results

In [ ]:
import pandas as pd

summary = pd.DataFrame(results).T
summary.index.name = "opponent"
summary

In [ ]:
import matplotlib.pyplot as plt

STARTING_MONEY = config.get("startingMoney", 3000)

fig, ax = plt.subplots(figsize=(8, 4.5))
opponents = list(summary.index)
means = summary["mean"]
errors = summary["stdev"]

ax.bar(opponents, means, yerr=errors, capsize=6, color="#4C72B0", edgecolor="white")
ax.axhline(
    STARTING_MONEY,
    color="#C44E52",
    linestyle="--",
    linewidth=1.2,
    label=f"Starting balance ({STARTING_MONEY:,})",
)

ax.set_title("Closing bank balance by opponent", fontsize=13, pad=12)
ax.set_ylabel("Bank balance")
ax.set_xlabel("Opponent")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

The dashed line marks the opening balance. It is the floor that matters: an agent
finishing below it has destroyed value relative to doing nothing at all, which an
earlier revision of this agent genuinely did on unlucky seeds.

## 7. Behavioural diagnostics

Aggregate scores say whether a change helped. They do not say why. For that we count
how the agent actually spends its turns, and inspect the farm it leaves behind at the
end of the season.

In [ ]:
def profile_episode(agent_path, opponent, seed):
    """Run one episode and record how turns and market orders were spent."""
    env = make(
        "kaggriculture",
        configuration={"episodeSteps": 720, "seed": seed},
        debug=False,
    )
    env.run([agent_path, opponent])

    unit_actions = Counter()
    market_orders = Counter()
    for step in env.steps:
        action = step[0].get("action") or {}
        farmer = action.get("farmer") or []
        if farmer:
            unit_actions[farmer[0]] += 1
        for hand in action.get("hands") or []:
            if hand:
                unit_actions[hand[0]] += 1
        for order in action.get("market") or []:
            if order:
                market_orders[order[0]] += 1

    final = env.steps[-1][0]
    return {
        "balance": final.reward,
        "unit_actions": unit_actions,
        "market_orders": market_orders,
        "final_farm": final.observation["farms"][0],
    }


profile = profile_episode(AGENT_PATH, "starter", seed=0)

print(f"Closing balance: {profile['balance']:,.0f}\n")
print("Market orders issued:")
for order, count in profile["market_orders"].most_common():
    print(f"  {order:12s} {count:4d}")

print("\nFarm at season end:")
for kind, count in describe_tiles(profile["final_farm"]).most_common():
    print(f"  {kind:12s} {count:4d}")

In [ ]:
actions = profile["unit_actions"].most_common(10)
labels = [name for name, _ in actions][::-1]
values = [count for _, count in actions][::-1]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(labels, values, color="#55A868", edgecolor="white")

ax.set_title("Turns by action type (farmer and hired hands)", fontsize=13, pad=12)
ax.set_xlabel("Turns")
ax.spines[["top", "right"]].set_visible(False)

for index, value in enumerate(values):
    ax.text(value, index, f" {value:,}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## 8. Findings

Three measurements changed how the agent is built. Each was found through the
diagnostics above rather than by reasoning about the rules.

**Tile upkeep capacity gates income, not sell prices.** An early revision planted more
tiles than one farmer could water. Plants weeded out, and because the agent never
issued `DIG`, each weeded tile stayed dead for the rest of the season. The farm decayed
to 23 of 25 tiles dead and sales starved to under four `SELL` orders per season. Adding
`DIG` moved every metric at once. Before optimising thresholds, check how many tiles
are still alive at season end.

**Hiring is the highest-return mechanic available.** The n-th hire of a day costs
`farmHandCostMult * fib(n)` and the counter resets each morning, so four hands cost
1 + 1 + 2 + 3 = 7 per day, roughly 210 for a full season. That expenditure is worth
well over a thousand in closing balance, because extra units raise exactly the upkeep
ceiling identified above. Hands are cleared nightly and must be re-hired each morning.

**Multiple units need explicit coordination.** Every unit running the same
"nearest useful tile" rule independently sends the whole crew to a single tile. Units
must reserve their targets so the work spreads across the farm.

## 9. Next steps

Ordered by expected value:

1. **`BUY_LAND`.** The crew can now maintain more tiles than the opening quadrant
   provides. Quadrants cost 1,000 / 2,000 / 4,000, so the question is when the marginal
   tile repays that outlay within a 30-day season.
2. **Animals.** Geese, cows, and sheep yield indefinitely while fed and never decay
   into weeds, but they require a wheat supply and daily feeding. This needs a
   grow-your-own-feed subsystem to be worth it.
3. **`FERTILIZE`.** Doubles the yield bonus for three days, but only on days the plant
   is also watered. Highest return on high-value, reliably tended plants.
4. **Sell scheduling.** Premium goods collapse toward the price floor on oversupply,
   and both players' orders clear one unit at a time against a shared market. Spreading
   sales should beat dumping a harvest in one order.